In [0]:
WITH base AS (
    SELECT 
        u.UserID,
        u.Name,
        u.Surname,
        u.Email,
        u.Gender,
        u.Age,
        u.Province,
        v.Channel2,
        v.RecordDate2
    FROM workspace.data.bright_tv_username u
    LEFT JOIN workspace.data.bright_tv_viewership v 
        ON u.UserID = v.UserID0
),

-- Total stats per user
user_stats AS (
    SELECT 
        UserID,
        COUNT(*) AS TotalViews,
        COUNT(DISTINCT Channel2) AS ChannelsWatched,
        MIN(RecordDate2) AS FirstWatchDate,
        MAX(RecordDate2) AS LastWatchDate,

        SUM(CASE WHEN dayofweek(RecordDate2) BETWEEN 2 AND 6 THEN 1 ELSE 0 END) AS WeekdayViews,
        SUM(CASE WHEN dayofweek(RecordDate2) IN (1,7) THEN 1 ELSE 0 END) AS WeekendViews
    FROM base
    GROUP BY UserID
),

-- Favorite channel using window function
fav_channel AS (
    SELECT UserID, Channel2
    FROM (
        SELECT 
            UserID,
            Channel2,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY UserID ORDER BY COUNT(*) DESC) AS rn
        FROM base
        GROUP BY UserID, Channel2
    ) t
    WHERE rn = 1
),

-- Peak viewing day
peak_day AS (
    SELECT UserID, RecordDate2
    FROM (
        SELECT 
            UserID,
            RecordDate2,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY UserID ORDER BY COUNT(*) DESC) AS rn
        FROM base
        GROUP BY UserID, RecordDate2
    ) t
    WHERE rn = 1
)

SELECT 
    u.UserID,
    u.Name,
    u.Surname,
    u.Email,
    u.Gender,
    u.Age,
    u.Province,

    -- Age group
    CASE 
        WHEN u.Age < 20 THEN '<20'
        WHEN u.Age BETWEEN 20 AND 29 THEN '20-29'
        WHEN u.Age BETWEEN 30 AND 39 THEN '30-39'
        ELSE '40+'
    END AS AgeGroup,

    s.TotalViews,
    s.ChannelsWatched,
    s.FirstWatchDate,
    s.LastWatchDate,
    s.WeekdayViews,
    s.WeekendViews,

    f.Channel2 AS FavoriteChannel,
    p.RecordDate2 AS PeakViewingDay,

    -- Activity level
    CASE 
        WHEN s.TotalViews >= 50 THEN 'High'
        WHEN s.TotalViews BETWEEN 20 AND 49 THEN 'Medium'
        ELSE 'Low'
    END AS ActivityLevel,

    -- Engagement score
    s.TotalViews * s.ChannelsWatched AS EngagementScore

FROM workspace.data.bright_tv_username u
LEFT JOIN user_stats s ON u.UserID = s.UserID
LEFT JOIN fav_channel f ON u.UserID = f.UserID
LEFT JOIN peak_day p ON u.UserID = p.UserID;